In [ ]:
# ============================================================
# CELL 1: All imports for Phase 3
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# All 8 classifiers from the paper
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Evaluation tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay)

plt.style.use('seaborn-v0_8-whitegrid')
print(" All imports successful!")

In [ ]:
# ============================================================
# CELL 2: Load the clean dataset we saved in Phase 2
# ============================================================

X = pd.read_csv('../data/processed/X_features.csv')
y = pd.read_csv('../data/processed/y_target.csv').squeeze()
# .squeeze() converts the single-column DataFrame into a simple Series

print(f"Features (X) shape : {X.shape}")
print(f"Target   (y) shape : {y.shape}")
print(f"\nTarget value counts:")
print(f"  Phishing   (1) : {(y==1).sum():,}")
print(f"  Legitimate(-1) : {(y==-1).sum():,}")
print("\n Dataset loaded!")

In [ ]:
# ============================================================
# CELL 3: Split data into 80% training and 20% testing
#         This matches Section 4.5 of the research paper exactly
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,              # our 30 features
    y,              # our target labels
    test_size=0.2,  # 20% for testing
    random_state=42 # fixes the randomness so results are reproducible
)

print(f"Training set size : {len(X_train):,} samples (80%)")
print(f"Testing  set size : {len(X_test):,}  samples (20%)")
print(f"\nTraining class distribution:")
print(f"  Phishing   : {(y_train==1).sum():,}")
print(f"  Legitimate : {(y_train==-1).sum():,}")
print(f"\nTesting class distribution:")
print(f"  Phishing   : {(y_test==1).sum():,}")
print(f"  Legitimate : {(y_test==-1).sum():,}")

In [ ]:
# ============================================================
# CELL 4: Define all 8 classifiers exactly as used in the paper
# ============================================================

models = {
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,    # number of trees to build
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,    # number of trees
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5       # look at 5 nearest neighbors to decide class
    ),
    'SVM': SVC(
        probability=True,   # we need this ON to calculate ROC-AUC score
        random_state=42
    ),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,      # give it enough iterations to converge
        random_state=42
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=100,
        random_state=42
    )
}

print(f" {len(models)} models defined and ready to train:")
for name in models:
    print(f"   - {name}")

In [ ]:
# ============================================================
# CELL 5: Train every model and collect all evaluation metrics
#         This is the core of Phase 3 — might take 2-3 minutes
# ============================================================

results = {}  # we'll store all metrics here

print("Training models... please wait")
print("=" * 55)

for name, model in models.items():
    print(f" Training {name}...", end=' ')
    
    # --- TRAIN ---
    model.fit(X_train, y_train)
    
    # --- PREDICT ---
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    # predict_proba gives confidence scores, we take column 1 (phishing probability)
    
    # --- EVALUATE ---
    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=1)
    recall    = recall_score(y_test, y_pred, pos_label=1)
    f1        = f1_score(y_test, y_pred, pos_label=1)
    mcc       = matthews_corrcoef(y_test, y_pred)
    auc       = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'Accuracy' : round(accuracy * 100, 1),
        'Precision': round(precision * 100, 1),
        'Recall'   : round(recall * 100, 1),
        'F1 Score' : round(f1 * 100, 1),
        'MCC'      : round(mcc, 2),
        'AUC'      : round(auc, 4)
    }
    
    print(f" Done! Accuracy: {accuracy*100:.1f}%")

print("\n All models trained successfully!")

In [ ]:
# ============================================================
# CELL 6: Display results as a clean table matching Table 4
#         in the research paper
# ============================================================

results_df = pd.DataFrame(results).T
# .T means transpose — flips rows and columns so models are rows

# Sort by Accuracy descending (best model on top)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("=" * 75)
print("MODEL PERFORMANCE COMPARISON TABLE (matches Table 4 in paper)")
print("=" * 75)
display(results_df.style
    .format({'Accuracy': '{:.1f}%',
             'Precision': '{:.1f}%',
             'Recall': '{:.1f}%',
             'F1 Score': '{:.1f}%',
             'MCC': '{:.2f}',
             'AUC': '{:.4f}'})
    .background_gradient(cmap='RdYlGn', subset=['Accuracy', 'F1 Score', 'MCC'])
    .set_caption("Green = better performance, Red = worse performance")
)

print("\n Paper's Table 4 results for comparison:")
print("   GB: Accuracy=97.1%, Precision=96%, Recall=98%, F1=97%, MCC=0.94, AUC=0.9959")
print("   Your results should be very close to these numbers!")

In [ ]:
# ============================================================
# CELL 7: Bar chart comparing accuracy of all 8 models
#         (matches Figure 11 in the paper)
# ============================================================

fig, ax = plt.subplots(figsize=(13, 6))

# Colors: gold for best model, blue for rest
colors = ['#f39c12' if name == 'Gradient Boosting' 
          else '#3498db' for name in results_df.index]

bars = ax.bar(
    results_df.index,
    results_df['Accuracy'],
    color=colors,
    edgecolor='black',
    linewidth=0.8,
    width=0.6
)

# Add value labels on top of each bar
for bar, val in zip(bars, results_df['Accuracy']):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f'{val:.1f}%',
        ha='center',
        fontweight='bold',
        fontsize=10
    )

ax.set_ylim(50, 105)
ax.set_xlabel('Machine Learning Algorithm', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy Comparison of All 8 ML Algorithms\n(Gold = Best Model: Gradient Boosting)',
             fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/processed/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Chart saved!")

In [ ]:
# ============================================================
# CELL 8: Compare ALL metrics side by side for all models
#         (matches Figure 16 in the paper)
# ============================================================

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x       = np.arange(len(results_df.index))
width   = 0.2  # width of each bar
colors  = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(15, 7))

for i, (metric, color) in enumerate(zip(metrics, colors)):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, results_df[metric],
                  width, label=metric, color=color,
                  edgecolor='black', linewidth=0.5)

ax.set_xlabel('Algorithm', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Full Metrics Comparison — All 8 Algorithms\n(Matches Figure 16 in Research Paper)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=15, ha='right')
ax.set_ylim(0, 115)
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig('../data/processed/full_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Chart saved!")

In [ ]:
# ============================================================
# CELL 9: ROC Curve for the best model (Gradient Boosting)
#         Matches Figure 17 in the paper
# ============================================================

# Get GB model predictions
gb_model     = models['Gradient Boosting']
y_pred_proba = gb_model.predict_proba(X_test)[:, 1]

# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba, pos_label=1)
auc_score            = roc_auc_score(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(8, 7))

# Plot the ROC curve
ax.plot(fpr, tpr,
        color='#2980b9',
        linewidth=2.5,
        label=f'GB — ROC curve (AUC = {auc_score:.4f})')

# Plot the random guess line (a bad model would follow this)
ax.plot([0, 1], [0, 1],
        color='gray',
        linewidth=1.5,
        linestyle='--',
        label='Random Guess (AUC = 0.50)')

ax.fill_between(fpr, tpr, alpha=0.1, color='#2980b9')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('ROC Curve — Gradient Boosting\n(Matches Figure 17 in Research Paper)',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('../data/processed/roc_curve_gb.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n GB AUC Score : {auc_score:.4f}")
print(f"   Paper reports: 0.9959")
print(f"   {' Great match!' if abs(auc_score - 0.9959) < 0.01 else ' Slight difference — this is normal'}")

In [ ]:
# ============================================================
# CELL 10: Confusion matrix for Gradient Boosting
#          Shows exactly where the model makes mistakes
# ============================================================

y_pred_gb = gb_model.predict(X_test)
cm        = confusion_matrix(y_test, y_pred_gb, labels=[1, -1])

fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Phishing', 'Legitimate']
)
disp.plot(ax=ax, colorbar=True, cmap='Blues')

ax.set_title('Confusion Matrix — Gradient Boosting\n(Best Model)',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_gb.png', dpi=150, bbox_inches='tight')
plt.show()

# Print what the numbers mean in plain English
tn, fp, fn, tp = cm[1][1], cm[1][0], cm[0][1], cm[0][0]
print(f"\n Reading the confusion matrix:")
print(f"    Phishing caught correctly  (True Positive) : {tp:,}")
print(f"    Legit identified correctly (True Negative) : {tn:,}")
print(f"    Legit wrongly flagged      (False Positive): {fp:,}")
print(f"    Phishing missed            (False Negative): {fn:,}")
print(f"\n   The most dangerous error is False Negative ({fn}) —")
print(f"   these are phishing sites the model missed completely!")

In [ ]:
# ============================================================
# CELL 11: Save the trained Gradient Boosting model to disk
#          We'll load this in Phase 4 without retraining
# ============================================================

import os
os.makedirs('../models', exist_ok=True)

# Save the model
joblib.dump(gb_model, '../models/gradient_boosting_model.pkl')

# Save the feature column names too — very important for Phase 4!
feature_names = list(X.columns)
joblib.dump(feature_names, '../models/feature_names.pkl')

# Save the full results table
results_df.to_csv('../data/processed/model_results.csv')

print(" Files saved:")
print("   - models/gradient_boosting_model.pkl  (trained model)")
print("   - models/feature_names.pkl            (feature column names)")
print("   - data/processed/model_results.csv    (results table)")

# Quick verification — load it back and test
loaded_model   = joblib.load('../models/gradient_boosting_model.pkl')
test_pred      = loaded_model.predict(X_test[:5])
print(f"\n Model reload verified!")
print(f"   Sample predictions: {test_pred}")
print(f"   (1=Phishing, -1=Legitimate)")